# Stream C - Evidence Validation (v3)

Checks and repairs missing evidence rows.

**Key change vs. previous version:** repair fallback no longer includes the caption in `evidence_text` (decircularized to match v3 evidence builder).


In [1]:
from pathlib import Path
import sys, os, json, pickle, re, warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd

def find_repo_root(start=None):
    p = Path(start or Path.cwd()).resolve()
    for q in [p, *p.parents]:
        if (q / "data").exists() and (q / "src").exists():
            return q
    raise RuntimeError("Could not find repo root.")

ROOT = find_repo_root()
sys.path.insert(0, str(ROOT))

from src.utils.paths import STREAM_C_CACHE, STREAM_A_CACHE, STREAM_B_CACHE, ALL_MANIFEST, FIGURES, TABLES

STREAM_C_CACHE.mkdir(parents=True, exist_ok=True)
FIGURES.mkdir(parents=True, exist_ok=True)
TABLES.mkdir(parents=True, exist_ok=True)
print("ROOT:", ROOT)
print("STREAM_C_CACHE:", STREAM_C_CACHE)


ROOT: E:\eve-rag
STREAM_C_CACHE: E:\eve-rag\cache\stream_c


In [2]:
EVIDENCE_DIR = STREAM_C_CACHE / "evidence"
EVIDENCE_DIR.mkdir(parents=True, exist_ok=True)
manifest = pd.read_csv(ALL_MANIFEST, dtype={"pair_id":"string","caption":"string"}, low_memory=False)
evidence_path = EVIDENCE_DIR / "raw_evidence_all.csv"
if evidence_path.exists():
    evidence = pd.read_csv(evidence_path, dtype={"pair_id":"string","caption":"string","evidence_text":"string"}, low_memory=False)
else:
    evidence = pd.DataFrame(columns=["pair_id","dataset","split","label","caption","evidence_text","evidence_len","caption_len"])
print("Manifest rows:", len(manifest)); print("Evidence rows:", len(evidence))


Manifest rows: 97361
Evidence rows: 97361


In [3]:
# === Repair missing rows WITHOUT including caption in evidence_text ===
# Critical: the repair fallback must keep evidence_text independent of the caption,
# otherwise the stance similarity score becomes circular and meaningless.
existing = set(evidence["pair_id"].astype(str))
missing = manifest[~manifest["pair_id"].astype(str).isin(existing)].copy()
print("Missing evidence rows:", len(missing))
if len(missing):
    repair = missing[["pair_id", "dataset", "split", "label", "caption"]].copy()
    # Decircularized fallback: dataset+split only. Do NOT include the caption.
    repair["evidence_text"] = "Dataset record: dataset=" + repair["dataset"].astype(str) + "; split=" + repair["split"].astype(str)
    repair["evidence_len"] = repair["evidence_text"].str.len()
    repair["caption_len"] = repair["caption"].fillna("").astype(str).str.len()
    evidence = pd.concat([evidence, repair], ignore_index=True)

order = {pid: i for i, pid in enumerate(manifest["pair_id"].astype(str))}
evidence["_order"] = evidence["pair_id"].astype(str).map(order)
evidence = evidence.sort_values("_order").drop_duplicates("pair_id", keep="first").drop(columns=["_order"])
evidence["evidence_text"] = evidence["evidence_text"].fillna("").astype(str)

# Sanity check: no caption leakage
# Sanity check: confirm no caption appears as a LABELED field in evidence_text.
# Substring overlap with article title is fine (titles legitimately contain captions);
# what we're guarding against is the old bug where "Caption: {caption}" was injected.
def _has_labeled_caption_leak(row):
    ev = (row["evidence_text"] or "")
    return "Caption:" in ev  # the old bug's signature

leak_count = int(evidence.apply(_has_labeled_caption_leak, axis=1).sum())
print(f"Labeled-caption leak count (should be 0): {leak_count}")
assert leak_count == 0, "Circularity detected: 'Caption:' field is being injected into evidence_text"

# Informational: how many rows have caption as a substring of evidence (incidental, OK)
def _caption_substring_overlap(row):
    cap = (row["caption"] or "").strip()
    ev = (row["evidence_text"] or "")
    if not cap or len(cap) < 20:
        return False
    return cap[:50] in ev
incidental = int(evidence.apply(_caption_substring_overlap, axis=1).sum())
print(f"Incidental caption/title overlap (informational, OK): {incidental}")

print("Final evidence rows:", len(evidence))
print("Empty evidence rows:", int((evidence["evidence_text"].str.len() == 0).sum()))
evidence.to_csv(EVIDENCE_DIR / "raw_evidence_all.csv", index=False)
pickle.dump(evidence, open(EVIDENCE_DIR / "raw_evidence_all.pkl", "wb"), protocol=4)


Missing evidence rows: 0
Labeled-caption leak count (should be 0): 0
Incidental caption/title overlap (informational, OK): 25
Final evidence rows: 97361
Empty evidence rows: 0


In [4]:
summary = evidence.groupby(["dataset", "split"]).agg(
    rows=("pair_id", "count"),
    empty_evidence=("evidence_text", lambda s: int((s.fillna("").astype(str).str.len() == 0).sum())),
    mean_evidence_len=("evidence_text", lambda s: float(s.fillna("").astype(str).str.len().mean()))
).reset_index()
summary.to_csv(TABLES / "stream_c_evidence_validation.csv", index=False)
display(summary)
print("Evidence validation v3 complete.")


,dataset,split,rows,empty_evidence,mean_evidence_len
0,mmfakebench,test,10000,0,229.172100
1,mmfakebench,val,1000,0,223.354000
2,newsclippings,test,7264,0,280.159967
3,newsclippings,train,71072,0,281.738758
4,newsclippings,val,7024,0,279.451737
5,verite,test,1001,0,99.516484


Evidence validation v3 complete.
